In [1]:
import logging
import pandas as pd
import numpy as np

# Configure logging for pipeline transparency and debugging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class CommerceDataPipeline:
    def __init__(self, raw_data_path: str):
        self.raw_data_path = raw_data_path
        self.df = None

    def ingest_data(self):
        """Simulates data extraction from warehouse or streaming source."""
        logging.info("Starting data ingestion phase...")
        try:
            # For demonstration, creating a simulated DataFrame of merchant session milestones
            np.random.seed(42)
            data = {
                'session_id': [f"sess_{i}" for i in range(1000)],
                'step_completed': np.random.choice(['view_product', 'add_to_cart', 'checkout_start', 'purchase'], size=1000, p=[0.4, 0.3, 0.2, 0.1]),
                'load_time_ms': np.random.normal(350, 50, 1000),
                'is_error': np.random.choice([0, 1], size=1000, p=[0.95, 0.05])
            }
            self.df = pd.DataFrame(data)
            logging.info(f"Successfully ingested {len(self.df)} records.")
        except Exception as e:
            logging.error(f"Ingestion failed: {e}")
            raise

    def validate_and_transform(self) -> pd.DataFrame:
        """Applies data quality checks, handles missing values, and calculates conversion drop-offs."""
        logging.info("Running data validation and transformation rules...")

        # Guard clause for empty data
        if self.df is None or self.df.empty:
            raise ValueError("Dataset is empty. Run ingest_data() first.")

        # Data quality & anomaly check: Flag sessions with abnormal load times or server errors
        initial_count = len(self.df)
        clean_df = self.df[self.df['is_error'] == 0].copy()
        dropped_count = initial_count - len(clean_df)
        logging.info(f"Dropped {dropped_count} sessions due to system errors.")

        # Calculate funnel drop-off metrics
        funnel_counts = clean_df['step_completed'].value_counts().reset_index()
        funnel_counts.columns = ['step', 'count']

        logging.info("Transformation complete. Funnel summary generated.")
        return funnel_counts

if __name__ == "__main__":
    # Execute pipeline workflow
    pipeline = CommerceDataPipeline(raw_data_path="mock_store_events.csv")
    pipeline.ingest_data()
    summary_metrics = pipeline.validate_and_transform()
    print("\n--- Funnel Summary Metrics ---")
    print(summary_metrics)


--- Funnel Summary Metrics ---
             step  count
0    view_product    401
1     add_to_cart    272
2  checkout_start    175
3        purchase     95
